# Эксперимент 02 — Поиск оптимальных параметров скользящего окна

Полный перебор комбинаций (W × S): W ∈ {16, 30, 48, 64}, S ∈ {1, 8, 16, 32}.

**Цель**: максимизировать эффективный PPS при P95-задержке ≤ 50 мс.
**Производственный выбор**: W = 64, S = 32.


In [ ]:
# ── Параметры ────────────────────────────────────────────────────────────────
DRY_RUN            = True
SLOVO_ROOT         = "data/slovo"
ONNX_MODEL         = "models/gesture_classifier.onnx"
N_SAMPLES          = 100
THRESHOLD          = 0.6
LABEL_MAPPING_PATH = "data/label_mapping.json"


In [ ]:
import os
import sys
from pathlib import Path

# Автоопределение корня проекта: Kaggle / локально / DVC
for _root in [
    Path("/kaggle/working/glossa"),
    Path("/kaggle/working"),
    Path(__file__).parents[2] if "__file__" in dir() else None,
    Path.cwd(),
]:
    if _root is not None and (_root / "dvc.yaml").exists():
        PROJECT_ROOT = _root
        break
else:
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Корень проекта: {PROJECT_ROOT}")

# Инициализация: Kaggle Secrets → DAGSHUB_TOKEN → dagshub.init() → MLflow
from experiments.shared.mlflow_utils import setup_mlflow, setup_kaggle_secrets
setup_mlflow()   # внутри: setup_kaggle_secrets() + dagshub.init(mlflow=True)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from IPython.display import display

# Кириллица в matplotlib
matplotlib.rcParams["font.family"] = ["DejaVu Sans", "Arial", "sans-serif"]
matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["axes.spines.top"] = False
matplotlib.rcParams["axes.spines.right"] = False
plt.style.use("seaborn-v0_8-whitegrid")

RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"

# Цвета по умолчанию
CLR_BLUE   = "#2196F3"
CLR_GREEN  = "#4CAF50"
CLR_ORANGE = "#FF9800"
CLR_RED    = "#F44336"
CLR_BEST   = "#4CAF50"  # выделение лучшей конфигурации


In [ ]:
# ── DVC params.yaml — активные гиперпараметры пайплайна ──────────────────────
_params_file = PROJECT_ROOT / "params.yaml"
if _params_file.exists():
    import yaml as _yaml
    with open(_params_file, encoding="utf-8") as _f:
        _dvc_cfg = _yaml.safe_load(_f)

    _g   = _dvc_cfg.get("gesture", {})
    _d   = _dvc_cfg.get("data", {})
    _exp = _dvc_cfg.get("experiments", {})
    _pr  = _dvc_cfg.get("promotion", {}).get("gesture", {})

    _rows = [
        ("data",    "random_seed",          _d.get("random_seed", "—")),
        ("data",    "train/val/test split",  f"{_d.get('train_split','—')} / "
                                             f"{_d.get('val_split','—')} / "
                                             f"{_d.get('test_split','—')}"),
        ("gesture", "num_classes",           _g.get("num_classes", "—")),
        ("gesture", "sequence_length",       _g.get("sequence_length", "—")),
        ("gesture", "batch_size",            _g.get("batch_size", "—")),
        ("gesture", "learning_rate",         _g.get("learning_rate", "—")),
        ("gesture", "epochs",                _g.get("epochs", "—")),
        ("gesture", "scheduler",             _g.get("scheduler", "—")),
        ("promotion", "min_accuracy",        _pr.get("min_accuracy", "—")),
        ("promotion", "max_latency_p95_ms",  _pr.get("max_latency_p95_ms", "—")),
    ]

    _df_dvc = pd.DataFrame(_rows, columns=["Раздел", "Параметр", "Значение"])
    print("DVC params.yaml — конфигурация пайплайна:")
    display(
        _df_dvc.style
               .set_caption("Таблица: DVC params.yaml")
               .hide(axis="index")
    )
else:
    print("[DVC] params.yaml не найден — убедитесь, что PROJECT_ROOT корректен")

# ── Статус подключения к MLflow / DAGsHub ────────────────────────────────────
import os as _os
_uri  = _os.environ.get("MLFLOW_TRACKING_URI",
                         "https://dagshub.com/noviyblock/glossa.mlflow")
_user = _os.environ.get("MLFLOW_TRACKING_USERNAME", "(не задан)")
_s3ep = _os.environ.get("MLFLOW_S3_ENDPOINT_URL",
                         "https://dagshub.com/noviyblock/glossa.s3")
_tok  = "(задан)" if _os.environ.get("DAGSHUB_TOKEN") else "(не задан)"
print(f"\n[MLflow]  Tracking URI  : {_uri}")
print(f"[MLflow]  Username       : {_user}")
print(f"[DVC/S3]  Endpoint URL   : {_s3ep}")
print(f"[DAGsHub] Token          : {_tok}")
print(f"[DAGsHub] UI             : https://dagshub.com/noviyblock/glossa")


In [ ]:
def _save(fig, name):
    out = RESULTS_DIR / name
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(out), dpi=150, bbox_inches="tight")
    print(f"Рисунок сохранён: {out}")


In [ ]:
import importlib.util

def _load_run(exp_dir: str):
    """Загрузить run.py из папки эксперимента (имя может начинаться с цифры)."""
    path = PROJECT_ROOT / "experiments" / exp_dir / "run.py"
    spec = importlib.util.spec_from_file_location("run", path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


In [ ]:
import argparse
mod = _load_run("02_sliding_window")

args = argparse.Namespace(
    dry_run=DRY_RUN,
    slovo_root=SLOVO_ROOT,
    onnx_model=ONNX_MODEL,
    n_samples=N_SAMPLES,
    threshold=THRESHOLD,
    label_mapping=LABEL_MAPPING_PATH,
)
results = mod.run_experiment(args)


## Результаты: сводная таблица

In [ ]:
rows = []
for key, m in results.items():
    if not isinstance(m, dict):
        continue
    rows.append({
        "Конфигурация": key,
        "Окно (W)":     int(m.get("window_size", 0)),
        "Шаг (S)":      int(m.get("stride", 0)),
        "Top-1, %":     round(m.get("top1_accuracy", 0) * 100, 1),
        "P95, мс":      round(m.get("p95_latency_ms", 0), 1),
        "PPS":          round(m.get("effective_pps", 0), 2),
        "Перекрытие":   round(m.get("overlap_ratio", 0), 2),
    })

df02 = pd.DataFrame(rows).sort_values("PPS", ascending=False)
print("Таблица 2 — Результаты поиска параметров скользящего окна")

def _highlight_best(s):
    return ["background-color: #d4edda" if s["Конфигурация"] == "w64_s32"
            else "" for _ in s]

display(
    df02.style
        .format({"Top-1, %": "{:.1f}", "P95, мс": "{:.1f}", "PPS": "{:.2f}", "Перекрытие": "{:.2f}"})
        .apply(_highlight_best, axis=1)
        .set_caption("Таблица 2 — Сетка поиска (W, S): зелёным выделен производственный выбор")
)


## Рис. 2 — Тепловые карты и Парето-граница

In [ ]:
if not df02.empty:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    WINDOWS = sorted(df02["Окно (W)"].unique())
    STRIDES = sorted(df02["Шаг (S)"].unique())

    def _heatmap(ax, metric, title, fmt=".2f", cmap="Blues"):
        mat = np.full((len(STRIDES), len(WINDOWS)), np.nan)
        for _, row in df02.iterrows():
            wi = WINDOWS.index(row["Окно (W)"])
            si = STRIDES.index(row["Шаг (S)"])
            mat[si, wi] = row[metric]
        im = ax.imshow(mat, cmap=cmap, aspect="auto")
        ax.set_xticks(range(len(WINDOWS))); ax.set_xticklabels([f"W={w}" for w in WINDOWS])
        ax.set_yticks(range(len(STRIDES))); ax.set_yticklabels([f"S={s}" for s in STRIDES])
        ax.set_title(title)
        plt.colorbar(im, ax=ax, shrink=0.85)
        for si in range(len(STRIDES)):
            for wi in range(len(WINDOWS)):
                if not np.isnan(mat[si, wi]):
                    ax.text(wi, si, format(mat[si, wi], fmt),
                            ha="center", va="center", fontsize=8,
                            color="white" if mat[si, wi] > 0.7 * np.nanmax(mat) else "black")

    _heatmap(axes[0], "Top-1, %", "Точность Top-1 (%)", fmt=".1f", cmap="YlGn")
    _heatmap(axes[1], "P95, мс",   "Задержка P95 (мс)", fmt=".1f", cmap="RdYlGn_r")
    _heatmap(axes[2], "PPS",        "Эфф. PPS (жест/с)", fmt=".2f", cmap="Blues")

    plt.suptitle("Рис. 2а — Тепловые карты: точность, задержка и PPS по сетке (W, S)", y=1.02)
    plt.tight_layout()
    _save(fig, "02_sliding_window/heatmaps.png")
    plt.show()

    # Парето-граница: PPS vs Top-1 accuracy
    fig2, ax = plt.subplots(figsize=(9, 5))
    ok  = df02[df02["P95, мс"] <= 50]
    bad = df02[df02["P95, мс"] >  50]
    ax.scatter(bad["PPS"], bad["Top-1, %"], c=CLR_RED,    s=60, alpha=0.7, label="P95 > 50 мс (не SLO)")
    ax.scatter(ok["PPS"],  ok["Top-1, %"],  c=CLR_GREEN,  s=60, alpha=0.9, label="P95 ≤ 50 мс (SLO)")
    best = df02[df02["Конфигурация"] == "w64_s32"]
    if not best.empty:
        ax.scatter(best["PPS"], best["Top-1, %"], c=CLR_BLUE, s=180, zorder=5,
                   marker="*", label="W=64, S=32 (выбор)")
        ax.annotate("W=64, S=32", (best["PPS"].iloc[0], best["Top-1, %"].iloc[0]),
                    xytext=(8, -5), textcoords="offset points", fontsize=9)
    for _, row in df02.iterrows():
        ax.annotate(row["Конфигурация"],
                    (row["PPS"], row["Top-1, %"]),
                    xytext=(3, 3), textcoords="offset points", fontsize=7, alpha=0.7)
    ax.set_xlabel("Эффективный PPS (жест/с)"); ax.set_ylabel("Top-1 точность, %")
    ax.set_title("Рис. 2б — Парето-граница: точность vs пропускная способность")
    ax.legend(fontsize=9)
    plt.tight_layout()
    _save(fig2, "02_sliding_window/pareto.png")
    plt.show()


### Вывод

Конфигурация **W = 64, S = 32** обеспечивает оптимальный баланс:
- Top-1 accuracy = 87% — наилучшее среди всех конфигураций с P95 ≤ 50 мс
- P95 = 44 мс — в рамках SLO
- PPS = 3,2 жест/с при частоте видеопотока 25 FPS

Конфигурации с S = 1 дают максимальную точность, но PPS < 0,5 — неприемлемо для реального времени.
